# Week 1 – Data Cleaning: Titanic Dataset

**Objective:** Clean the Titanic passenger dataset (891 rows, mix of numeric and categorical columns) by handling missing values, duplicates, and incorrect data types, as the first step before visualization.

Dataset source: [Titanic dataset, public GitHub mirror of the classic Kaggle competition dataset](https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv)

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('titanic.csv')
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


## Step 1: Inspect the data

Before cleaning anything, look at the shape, column types, and non-null counts.

In [2]:
print(df.shape)
df.info()

(891, 12)
<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 83.7 KB


## Step 2: Check for missing values

`df.info()` already hints at missing data (columns with fewer non-null values than the total row count). `isnull().sum()` gives exact counts per column.

In [3]:
df.isnull().sum()

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

**What we found:**
- `Age` is missing for 177 of 891 passengers (~20%) — too many to drop, so we'll impute.
- `Cabin` is missing for 687 of 891 passengers (~77%) — too sparse to reliably impute, so we'll drop the column instead of the rows.
- `Embarked` is missing for only 2 passengers — safe to drop those rows or fill with the most common port.

**Reasoning for how each will be handled:**
- `Age`: fill missing values with the **median** age. Median is more robust than the mean here since age has some outliers (e.g. infants and elderly passengers), and using a single central value preserves all 891 rows for later analysis.
- `Cabin`: drop the column entirely. With over three-quarters of values missing, any imputation would be mostly guesswork and could mislead visualizations.
- `Embarked`: fill the 2 missing values with the **mode** (most frequent port), since it's categorical and only 2 rows are affected — dropping them would lose negligible information, but filling keeps the dataset fully intact.

In [4]:
# Fill missing Age with the median
df['Age'] = df['Age'].fillna(df['Age'].median())

# Drop Cabin column - too many missing values to impute reliably
df = df.drop(columns=['Cabin'])

# Fill missing Embarked with the mode (most common port)
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])

# Confirm no missing values remain
df.isnull().sum()

PassengerId    0
Survived       0
Pclass         0
Name           0
Sex            0
Age            0
SibSp          0
Parch          0
Ticket         0
Fare           0
Embarked       0
dtype: int64

## Step 3: Check for and remove duplicates

Duplicate rows can skew visualizations and statistics, so we check for and remove any exact duplicate records.

In [5]:
print('Duplicate rows found:', df.duplicated().sum())
df = df.drop_duplicates()
print('Shape after dropping duplicates:', df.shape)

Duplicate rows found: 0
Shape after dropping duplicates: (891, 11)


**Note:** this dataset happened to have 0 exact duplicate rows, but running `drop_duplicates()` is still good practice — it's a required check even when it turns out to be a no-op, since new or larger datasets often do contain duplicates.

## Step 4: Fix incorrect data types

Some columns are stored as numbers but are actually categories (e.g. `Pclass` is a passenger class label: 1st/2nd/3rd, not a quantity; `Survived` is a yes/no flag, not a quantity). Converting these to proper categorical types makes later grouping and plotting clearer and prevents accidentally treating them as continuous numeric data.

In [6]:
# Convert categorical-looking numeric codes to actual categories
df['Survived'] = df['Survived'].map({0: 'No', 1: 'Yes'}).astype('category')
df['Pclass'] = df['Pclass'].astype('category')
df['Sex'] = df['Sex'].astype('category')
df['Embarked'] = df['Embarked'].astype('category')

df.dtypes

PassengerId       int64
Survived       category
Pclass         category
Name                str
Sex            category
Age             float64
SibSp             int64
Parch             int64
Ticket              str
Fare            float64
Embarked       category
dtype: object

## Step 5: Final check

The dataset is now clean:
- No missing values remain (Age imputed, Cabin dropped, Embarked imputed).
- No duplicate rows.
- Categorical columns (`Survived`, `Pclass`, `Sex`, `Embarked`) are properly typed instead of being left as raw numbers or generic objects.

This cleaned `df` is what we'll use in the next step to build the 3–5 visualizations.

In [7]:
df.info()
df.head()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype   
---  ------       --------------  -----   
 0   PassengerId  891 non-null    int64   
 1   Survived     891 non-null    category
 2   Pclass       891 non-null    category
 3   Name         891 non-null    str     
 4   Sex          891 non-null    category
 5   Age          891 non-null    float64 
 6   SibSp        891 non-null    int64   
 7   Parch        891 non-null    int64   
 8   Ticket       891 non-null    str     
 9   Fare         891 non-null    float64 
 10  Embarked     891 non-null    category
dtypes: category(4), float64(2), int64(3), str(2)
memory usage: 52.4 KB


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Embarked
0,1,No,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,S
1,2,Yes,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C
2,3,Yes,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,S
3,4,Yes,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,S
4,5,No,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,S


In [8]:
# Save the cleaned dataset for the visualization step
df.to_csv('titanic_cleaned.csv', index=False)